# Assignment 1: Gaussians, Categories, and Clusters

**Corresponding textbook chapters:** Tutorial 1 Ch 5 (Bayesian inference) and [Tutorial 3 Ch 5 — Mixture Models](https://josephausterweil.github.io/probintro/intro2/05_mixture_models/). (The mixture-models chapter lives in the *continuous* tutorial — Tutorial 3 in URL terms — even though the URL path is `intro2/`. Tutorial 2 = the GenJAX tutorial at `genjax/`; Tutorial 3 = the continuous-probability tutorial at `intro2/`.)

**This is the GenJAX (canonical) stencil.** Problem 1 (Gaussian-Gaussian conjugate) is closed-form, so we use `numpy` + `scipy.stats` for the math. Problem 2 builds toward a GenJAX generative mixture model in Part 2(e) — see Tutorial 2, Chapters 0–4 for background on `@gen`, `flip`, `normal`, and `simulate`.

**A note on Bernoulli in GenJAX:** the GenJAX library has both `bernoulli(p)` (which interprets `p` as a *logit*) and `flip(p)` (which interprets `p` as a *probability*). For this assignment we want `flip` — `flip(0.7)` produces True 70% of the time, while `bernoulli(0.7)` would produce True only ~67% of the time. Tutorial 2 Ch 2 has the full warning.

If you'd prefer a different language, see `clusters_python.ipynb` (Python without GenJAX, with optional paired "Now in GenJAX" cells) or `clusters_nosoln.Rmd` (R) in the same directory. Matlab stencil available on request — DM Joe.

---

## Setup

If running in Google Colab, run the cell below to install GenJAX. (On the first run of the session only.)

In [ ]:
!pip install genjax

In [ ]:
# Import packages
import jax
import jax.numpy as jnp
import jax.random as random
import jax.lax as lax
from genjax import gen, flip, normal   # NOTE: `flip(p)` takes a probability; `bernoulli(p)` takes a logit.
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

np.random.seed(42)
key = random.PRNGKey(42)

# NOTE on dtypes: GenJAX distributions produce float32. When you pass scalars
# into a GenJAX model (Part 2(e)), cast them with jnp.float32(...) so they do
# not mix with numpy's float64 — a dtype mismatch raises a TypeError inside the
# sampler. The Part 2(e) solution below does this.

---

# Problem 1: Gaussian-Gaussian Conjugate Model

We start with the **Gaussian-Gaussian** conjugate model: a Gaussian likelihood with unknown mean $\mu$ and *known* variance $\sigma_x^2$, with a Gaussian prior on $\mu$:

$$\mu \sim \mathcal{N}(\mu_0, \sigma_0^2) \qquad \qquad x_1, \dotsc, x_N \mid \mu, \sigma_x^2 \overset{iid}{\sim} \mathcal{N}(\mu, \sigma_x^2)$$

Because the prior is conjugate to the likelihood, the posterior and posterior-predictive distributions have closed forms:

$$\mu \mid x_1, \dotsc, x_N \sim \mathcal{N}\!\left(\frac{\mu_0 \sigma_0^{-2} + \sigma_x^{-2} \sum_n x_n}{\sigma_0^{-2} + N \sigma_x^{-2}},\; \left[\sigma_0^{-2} + N \sigma_x^{-2}\right]^{-1}\right)$$

$$x_{N+1} \mid x_1, \dotsc, x_N \sim \mathcal{N}\!\left(\text{same mean as posterior},\; \left[\sigma_0^{-2} + N \sigma_x^{-2}\right]^{-1} + \sigma_x^2\right)$$

The predictive has the same mean as the posterior but its variance is inflated by $\sigma_x^2$ — the irreducible noise of the likelihood.

**For Problem 1, use $\mu_0 = 0$ and $\sigma_0^2 = 1$.**

**Note on GenJAX.** Problem 1 is closed-form, so we will use `numpy` + `scipy.stats` here — there is nothing for GenJAX to add. GenJAX returns in Problem 2 Part (e), where we sample from a non-conjugate mixture model.


## Part 1(a): Prior plot

To provide a baseline, plot the prior distribution $p(\mu) = \mathcal{N}(\mu; \mu_0, \sigma_0^2)$ over a range that captures both tails and the peak.


In [ ]:
# fill me
#
# Suggested approach:
#   1. Pick mu_0 = 0, sigma_0_squared = 1.
#   2. Build mu_range with np.linspace(-4, 4, 1000) (captures ~4 std devs).
#   3. Evaluate the prior density with scipy.stats.norm.pdf(mu_range, mu_0, sqrt(sigma_0_squared)).
#   4. Plot density vs. mu_range; label the axes.

from scipy.stats import norm

mu_0 = 0.0
sigma_0_squared = 1.0

mu_range = np.linspace(-4, 4, 1000)

fig, ax = plt.subplots(figsize=(8, 5))

# your plotting code here

ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$p(\mu)$')
ax.set_title(r'Prior: $\mathcal{N}(\mu_0, \sigma_0^2)$')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Part 1(b): One-datum update

Calculate and plot the **posterior** $p(\mu \mid x_1)$ **and** the **posterior-predictive** $p(x_2 \mid x_1)$ after observing $x_1 = 2$, for $\sigma_x^2 = 0.25$ and $\sigma_x^2 = 4$. That is four distributions in total (posterior + predictive for each $\sigma_x^2$).

**Question.** How does changing the likelihood variance $\sigma_x^2$ affect the posterior and the predictive? Where are the two distributions similar? Where do they differ, and why?

**Hint.** A pre-written helper `conjugate_update` is provided below. It returns the posterior mean and variance, and the posterior-predictive variance. Fill in its body using the closed-form expressions above.


In [ ]:
def conjugate_update(mu_0, sigma_0_squared, sigma_x_squared, data):
    """
    Conjugate Gaussian-Gaussian update.

    Args:
        mu_0:               Prior mean of mu
        sigma_0_squared:    Prior variance of mu
        sigma_x_squared:    Known likelihood variance
        data:               1D np.array of observed x values (can be empty for prior)

    Returns:
        post_mean:          Posterior mean of mu given data
        post_var:           Posterior variance of mu given data
        pred_var:           Posterior-predictive variance for a new x (post_var + sigma_x_squared)
    """
    # fill me
    #
    # Suggested approach:
    #   1. Let N = len(data); sum_x = data.sum() (handle N=0 as the prior).
    #   2. Posterior precision = 1/sigma_0_squared + N / sigma_x_squared.
    #   3. Posterior mean    = (mu_0 / sigma_0_squared + sum_x / sigma_x_squared) / posterior_precision.
    #   4. Posterior variance = 1 / posterior_precision.
    #   5. Predictive variance = posterior variance + sigma_x_squared.
    pass


In [ ]:
# fill me
#
# Plot the posterior p(mu | x_1=2) and predictive p(x_2 | x_1=2)
# for sigma_x_squared in {0.25, 4}.

mu_0 = 0.0
sigma_0_squared = 1.0
x_1 = np.array([2.0])

sigma_x_squared_values = [0.25, 4.0]

# Plot range: wide enough to show both tails for both choices of sigma_x_squared.
plot_range = np.linspace(-4, 6, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, sigma_x_squared in zip(axes, sigma_x_squared_values):
    # 1. Call conjugate_update to get post_mean, post_var, pred_var.
    # 2. Plot the posterior   N(plot_range; post_mean, sqrt(post_var)).
    # 3. Plot the predictive  N(plot_range; post_mean, sqrt(pred_var)).
    # 4. Mark x_1 with a vertical line.

    # your code here

    ax.set_xlabel(r'value')
    ax.set_ylabel(r'density')
    ax.set_title(rf'$\sigma_x^2 = {sigma_x_squared}$')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


**Your answer.**

- Where are the posterior and predictive distributions *similar*, and where do they *differ*?
- What is the effect of increasing $\sigma_x^2$ from 0.25 to 4? Which distribution moves more? Why?

*(your answer here)*


## Part 1(c): Multiple-datum update

Now observe five data points: $(x_1, \dotsc, x_5) = (2.1, 2.5, 1.4, 2.2, 1.8)$. Plot the posterior and predictive for $\sigma_x^2 = 0.25$ and $\sigma_x^2 = 4$.

The average of these five points is exactly $2.0$, the same as the single datum in Part 1(b). **Compare** the resulting posteriors and predictives to Part 1(b).

**Question.** Where do the Part 1(b) and Part 1(c) results agree, and where do they differ? For the cases that differ, why? For the cases that don't, why not? (Hint: think about the role of $N$ in the posterior precision.)


In [ ]:
# fill me
#
# Reuse conjugate_update, but pass the full 5-point data array.

mu_0 = 0.0
sigma_0_squared = 1.0
data = np.array([2.1, 2.5, 1.4, 2.2, 1.8])

sigma_x_squared_values = [0.25, 4.0]

plot_range = np.linspace(-4, 6, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, sigma_x_squared in zip(axes, sigma_x_squared_values):
    # Same recipe as Part 1(b), but with the 5-point data array.

    # your code here

    ax.set_xlabel(r'value')
    ax.set_ylabel(r'density')
    ax.set_title(rf'$\sigma_x^2 = {sigma_x_squared},\ N = 5$')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


**Your answer.**

- How do the Part 1(c) posteriors compare to Part 1(b)? Where are they the same? Where do they differ?
- The data average is 2.0 in both cases, so the posterior *mean* should be similar. What about the posterior *variance*?
- What about the predictive variance? Is it dominated by the posterior variance or by $\sigma_x^2$?

*(your answer here)*


---

# Problem 2: Gaussian Mixture / Categorization

In this problem, we make **categorization decisions** for two categories, each defined as a Gaussian distribution. You will derive the probability of an item being in one category vs. the other, then explore how the variances and prior probability of each category affect the posterior and the predictive distribution.

Data are generated by first picking which of two categories $c = 1, 2$ a datum belongs to (according to their prior probability) and then generating the datum from the corresponding category's likelihood:

$$c_n | \theta \sim \text{Bernoulli}(\theta) \qquad \qquad x_n | \mu_{c(n)}, \sigma_{c(n)}^2 \overset{iid}{\sim} \mathcal{N}(\mu_{c(n)}, \sigma_{c(n)}^2)$$

Note that $c(n)$ is the same as $c_n$; the parentheses are used to avoid double-subscripts. $c_n = 1$ with probability $\theta$ (and $c_n = 2$ with probability $1 - \theta$), so the prior probability of category 1 is $\theta$: $P(c_n = 1) = \theta$.

**For all of Problem 2, assume $\mu_1 = -1$ and $\mu_2 = 1$.**

---

## Part 2(a): Derivation — Categorization

Using **Bayes' rule**, derive the probability of a single datum being in category 1: $P(c_1 = 1 | x_1)$. You can assume that the values of $\mu_1, \mu_2, \sigma_1^2,$ and $\sigma_2^2$ are given parameters. Show your work (if you do not know how to create equations on a computer, you can scan your handwritten derivation and include it as an image).

As the next problem depends on this answer, the derivation should end up with:

$$P(c_1 = 1 | x_1) = \frac{\theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2)}{\theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2) + (1 - \theta) \, \mathcal{N}(x_1; \mu_2, \sigma_2^2)}$$

where $\mathcal{N}(x; \mu, \sigma^2)$ is the probability density of $x$ from a Normal distribution with mean $\mu$ and variance $\sigma^2$.

**Grading note: no credit for transcribing the given answer. Show every step (Bayes' rule, expand the denominator with the Law of Total Probability).**

#### **Derivation**

*(fill in your derivation here — show your work)*

---

## Part 2(b): Categorization

Calculate and plot the probability of being in category 1 (x-axis is the $x_1$ value; y-axis is $P(c_1 = 1 | x_1)$) for:

1. $\theta = 0.5$ and $\theta = 0.75$, with $\sigma_1^2 = \sigma_2^2 = 1$.
2. $\theta = 0.5$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.
3. $\theta = 0.75$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.

Make sure your plots capture the interesting behavior (appropriate x- and y-axis ranges).

**Hint.** A pre-written helper function `update_datum_c1` is provided below. It returns the posterior $P(c_1 = 1 | x)$ and the marginal $p(x)$ over an x-range. You should fill in its body using the result from part (a) — but use GenJAX primitives (see the suggestion comments inside). Alternatively, you may use `scipy.stats.norm.pdf` if you prefer to compute analytically; the tutorial chapters demonstrate both.

**Question.** Describe the effect of changing the prior and the variance on categorization decisions. Do they have the same effect? Why or why not?

In [ ]:
def update_datum_c1(mu_1, mu_2, sigma_1_squared, sigma_2_squared, theta, x_min, x_max):
    """
    Compute posterior probability of category 1 and marginal distribution.

    Args:
        mu_1, mu_2: Means of categories 1 and 2
        sigma_1_squared, sigma_2_squared: Variances of categories 1 and 2
        theta: Prior probability of category 1
        x_min, x_max: Range over which to evaluate

    Returns:
        posterior_c1: P(c=1|x) evaluated over x_range
        marginal: p(x) evaluated over x_range
    """
    # fill me
    #
    # Suggested approach:
    #   1. Build x_range with np.linspace(x_min, x_max, 1000)
    #   2. Compute the two Gaussian likelihoods at each x in x_range
    #      (using scipy.stats.norm.pdf, or a jax.scipy equivalent)
    #   3. Apply Bayes' rule from part (a) to get posterior_c1
    #   4. Apply the Law of Total Probability to get marginal
    pass

In [ ]:
# fill me
#
# Loop over the configurations required in part (b) and plot
# the posterior P(c=1|x) for each (theta, sigma_1^2, sigma_2^2) combination.

# Fixed means (given)
mu_1 = -1.0
mu_2 = 1.0

# Configurations to plot
theta_values = [0.5, 0.75]
configs = [
    (1, 1),      # sigma_1^2 = sigma_2^2 = 1
    (0.5, 2),    # sigma_1^2 = 0.5, sigma_2^2 = 2
]

# Suggested plotting aids
colors = {0.5: "blue", 0.75: "orange"}
linestyles = {1: "-", 0.5: "--", 2: ":"}

x_min, x_max = -6, 6
x_range = np.linspace(x_min, x_max, 1000)

fig, ax = plt.subplots(figsize=(10, 6))

# your plotting loop here

ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$P(c_1 = 1 | x)$')
ax.set_title('Posterior categorization probability')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Describe the effect of changing the prior and the variance on categorization decisions.**

- What is the effect of varying them? Do they have the same effect?
- Why or why not?

*(your answer here)*

---

## Part 2(c): Derivation — Prediction

Using Bayes' rule and the **Law of Total Probability**, derive the probability of a data point $p(x_1)$ according to this model (without any given data). As the next problem depends on this answer, the derivation should end up with:

$$p(x_1) = \theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2) + (1 - \theta) \, \mathcal{N}(x_1; \mu_2, \sigma_2^2)$$

**Grading note: no credit for transcribing the given answer. Show every step (Law of Total Probability over $c$, then expand $p(x|c)P(c)$ for each category).**

#### **Derivation**

*(fill in your derivation here — show your work)*

---

## Part 2(d): Prediction

Plot $p(x_1)$ for:

1. $\theta = 0.5$ and $\theta = 0.75$, with $\sigma_1^2 = \sigma_2^2 = 1$.
2. $\theta = 0.5$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.
3. $\theta = 0.75$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.

Make sure your plots capture the interesting behavior.

Note that $p(x_1)$ is sometimes called the *marginal data distribution*; this type of model is called a **mixture model** because it composes a new distribution by "mixing" two (or more) distributions together.

**Question.** How does the prior and variance affect $p(x_1)$? Do they have the same effect? Why or why not?

In [ ]:
# fill me
#
# Loop over the configurations required in part (d) and plot
# the marginal p(x) for each (theta, sigma_1^2, sigma_2^2) combination.

theta_values = [0.5, 0.75]
configs = [
    (1, 1),
    (0.5, 2),
]

colors = {0.5: "blue", 0.75: "orange"}
linestyles = {1: "-", 0.5: "--"}

x_min, x_max = -6, 6
x_range = np.linspace(x_min, x_max, 1000)

fig, ax = plt.subplots(figsize=(10, 6))

# your plotting loop here

ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$p(x)$')
ax.set_title('Marginal (predictive) distribution')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Describe the effect of changing the prior and the variance on $p(x_1)$.**

- What is the effect of varying them? Do they have the same effect?
- Why or why not?

*(your answer here)*

---

## Part 2(e): GenJAX Mixture Model

Write a GenJAX generative model `gaussian_mixture_model(theta, mu_1, mu_2, sigma_1, sigma_2)` that implements the generative process from the Problem Setup. Your model should:

1. Sample a category $c$ from a Bernoulli with parameter $\theta$, named `"category"`. **Use `flip(theta)`, not `bernoulli(theta)`** — see the note at the top of the notebook.
2. Conditional on $c$, sample $x$ from the corresponding Gaussian, named `"observation"`.
3. Return the observation $x$ and category $c$.

Then simulate $N = 2000$ samples with $\theta = 0.7$, $\mu_1 = -1, \mu_2 = 1$, $\sigma_1 = \sigma_2 = 1$. Plot a histogram of the observations overlaid on the theoretical marginal $p(x)$ from part (c). Verify that the empirical distribution matches.

**Hint.** Look at Tutorial 2, Chapter 2 for how to structure a `@gen` function. For the conditional Gaussian (sample from category 1's or category 2's likelihood based on $c$), `jnp.where(c, mu_1, mu_2)` and `jnp.where(c, sigma_1, sigma_2)` work cleanly because `flip` returns a Boolean that casts to 0/1.

In [ ]:
# fill me

@gen
def gaussian_mixture_model(theta, mu_1, mu_2, sigma_1, sigma_2):
    """
    GenJAX generative model for a 2-component Gaussian mixture.
    """
    # 1. Sample category ~ flip(theta), addressed as "category".
    #    (flip(p) returns a Boolean True/False; True = category 1, False = category 2.)
    # 2. Compute mu_c = jnp.where(c, mu_1, mu_2) and sigma_c = jnp.where(c, sigma_1, sigma_2).
    # 3. Sample observation ~ normal(mu_c, sigma_c), addressed as "observation".
    # 4. Return (observation, category).
    pass


# Simulation loop: vmap simulate over N=2000 keys, extract "observation" from each trace,
# plot histogram (density=True) and overlay the analytical p(x) curve from Part 2(c).
#
# DTYPE: cast the scalar arguments you pass to gaussian_mixture_model with
# jnp.float32(...) — e.g. theta = jnp.float32(0.7). GenJAX distributions are
# float32, and mixing in a plain Python/numpy float64 raises a TypeError.
# your code here

---

## Submission

Submit this completed notebook (runs end-to-end with no errors) plus a short PDF with your derivations for parts (a) and (c) if you prefer to handwrite or typeset them separately.